## Import and setup

In [ ]:
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

csv_path = Path("../data/steam_description_data.csv")
df = pd.read_csv(csv_path)

df["about_the_game"] = df["about_the_game"].fillna("")
df["detailed_description"] = df["detailed_description"].fillna("")
df["combined_description"] = (df["about_the_game"] + " " + df["detailed_description"]).str.strip()

# Device selection
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

Device: mps


## Baseline model

In [10]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# pad token for gpt2
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

model.eval()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10043.47it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## Baseline Generation Prompt

In [11]:
baseline_prompts = [
    "A game about space exploration",
    "A game with multiplayer action",
    "A story-driven adventure game",
    "A puzzle game for kids",
    "A horror survival game",
]

gen_kwargs = {
    "max_new_tokens": 80,
    "temperature": 0.8,
    "top_k": 50,
    "top_p": 0.95,
    "do_sample": True,
    "num_return_sequences": 1,
    "pad_token_id": tokenizer.eos_token_id,
}

baseline_outputs = []


for i, prompt in enumerate(baseline_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = model.generate(**inputs, **gen_kwargs)
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    baseline_outputs.append({"prompt": prompt, "output": text})

    print(f"\n--- Baseline Output {i} ---")
    print(f"Prompt: {prompt}")
    print(f"Generated: {text}")


--- Baseline Output 1 ---
Prompt: A game about space exploration
Generated: A game about space exploration, but I'm going to take a quick look at the actual game from its inception, and how it can be used in your development.

What makes the game different from other games in that respect?

The first game has the same premise but with a slightly different approach to gameplay.

You're first going to explore the world on your own.

This game is an

--- Baseline Output 2 ---
Prompt: A game with multiplayer action
Generated: A game with multiplayer action. The game will be released in the spring of 2017 and will include the following:

An introduction to the game in a game with multiplayer action. The game will be released in the spring of 2017 and will include the following:

--- Baseline Output 3 ---
Prompt: A story-driven adventure game
Generated: A story-driven adventure game for the Xbox One. Get more info at http://www.xboxone.com/.

Developer: Microsoft Studios

Publisher: Microso

## Baseline Generation Time

In [14]:
import time
import numpy as np

latencies = []
idx_latency = 0


for prompt in baseline_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    start = time.perf_counter()
    _ = model.generate(**inputs, **gen_kwargs)
    end = time.perf_counter()

    latencies.append(end - start)

    print(latencies[idx_latency])
    idx_latency = idx_latency + 1

avg_latency = float(np.mean(latencies))
print(f"Average baseline generation time: {avg_latency:.4f} sec")

1.597947791997285
0.40043287499793223
1.4833107080012269
1.5503330839965201
1.4471429999975953
Average baseline generation time: 1.2958 sec


## Estimate model size and save outputs + metrics

In [17]:
from pathlib import Path
import pandas as pd

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

# baseline outputs
out_txt = results_dir / "baseline_outputs.txt"
with out_txt.open("w", encoding="utf-8") as f:
    for i, item in enumerate(baseline_outputs, 1):
        f.write(f"--- Baseline Output {i} ---\n")
        f.write(f"Prompt: {item['prompt']}\n")
        f.write(f"Generated: {item['output']}\n")

# Approx model size in MB (parameters float32)
num_params = sum(p.numel() for p in model.parameters())
model_size_mb = num_params * 4 / (1024 ** 2)

baseline_row = {
    "model": "gpt2-baseline",
    "avg_generation_time_sec": avg_latency,
    "max_new_tokens": gen_kwargs["max_new_tokens"],
    "num_return_sequences": gen_kwargs["num_return_sequences"],
    "approx_model_size_mb": model_size_mb,
}

metrics_path = results_dir / "metrics.csv"
if metrics_path.exists():
    old = pd.read_csv(metrics_path)
    metrics_df = pd.concat([old, pd.DataFrame([baseline_row])], ignore_index=True)
else:
    metrics_df = pd.DataFrame([baseline_row])

metrics_df.to_csv(metrics_path, index=False)

print(f"Saved outputs to: {out_txt}")
print(f"Saved metrics to: {metrics_path}")
print(baseline_row)

Saved outputs to: ../results/baseline_outputs.txt
Saved metrics to: ../results/metrics.csv
{'model': 'gpt2-baseline', 'avg_generation_time_sec': 1.295833491798112, 'max_new_tokens': 80, 'num_return_sequences': 1, 'approx_model_size_mb': 474.7001953125}


## Dataset Load

In [16]:
import pandas as pd
from datasets import Dataset

data_path = "../data/processed/steam_games_cleaned.csv"  # adjust if your filename differs
df = pd.read_csv(data_path)

print("Shape:", df.shape)
display(df.head(3))
print("\nColumns:", df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/steam_games_cleaned.csv'